# Monitoring Setup Blueprint (Aufgabe 6)

Dieses Notebook enthaelt die Anleitung und Ausfuehrung fuer den Monitoring-Stack in diesem Projekt.

## Branch-Check
- Aktueller Branch: `feature/application_monitoring`
- Damit arbeitest du **nicht** auf `main`.

## Ziel
Runtime-Monitoring fuer die Inference-Pipeline mit:
- Zenoh
- Detector-Service (Metriken)
- App-Service (API + Metriken)
- Prometheus
- Grafana

## Enthaltene Dateien

Der Generator schreibt folgende Dateien:
- `docker-compose.monitoring.yml`
- `Dockerfile.detector`
- `Dockerfile.app`
- `runtime/detector_service.py`
- `runtime/app_service.py`
- `monitoring/prometheus/prometheus.yml`
- `monitoring/grafana/provisioning/datasources/datasources.yml`
- `monitoring/grafana/provisioning/dashboards/dashboards.yml`
- `monitoring/grafana/dashboards/litter-overview.json`
- `docs/monitoring.md`

Manueller Schritt in `pyproject.toml` (dependencies):
- `fastapi>=0.115.0`
- `uvicorn>=0.30.0`
- `prometheus-client>=0.24.0`
- `eclipse-zenoh>=1.0.0`

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

FILES = {
    "docker-compose.monitoring.yml": """version: "3.9"

services:
  zenoh:
    image: eclipse/zenoh:latest
    command: ["-l", "tcp/0.0.0.0:7447"]
    ports:
      - "7447:7447"

  detector:
    build:
      context: .
      dockerfile: Dockerfile.detector
    environment:
      ZENOH_ROUTER: "zenoh:7447"
      CAMERA_MODE: "synthetic"
      DETECTOR_LOOP_HZ: "8"
      PROMETHEUS_PORT: "9100"
      MODEL_NAME: "synthetic-unet"
    depends_on:
      - zenoh
    ports:
      - "9100:9100"

  app:
    build:
      context: .
      dockerfile: Dockerfile.app
    environment:
      ZENOH_ROUTER: "zenoh:7447"
      APP_PORT: "8000"
      PROMETHEUS_PORT: "9101"
    depends_on:
      - zenoh
    ports:
      - "8000:8000"
      - "9101:9101"

  prometheus:
    image: prom/prometheus:latest
    volumes:
      - ./monitoring/prometheus/prometheus.yml:/etc/prometheus/prometheus.yml:ro
    ports:
      - "9090:9090"
    depends_on:
      - detector
      - app

  grafana:
    image: grafana/grafana:latest
    depends_on:
      - prometheus
    ports:
      - "3000:3000"
    volumes:
      - ./monitoring/grafana/provisioning/datasources:/etc/grafana/provisioning/datasources:ro
      - ./monitoring/grafana/provisioning/dashboards:/etc/grafana/provisioning/dashboards:ro
      - ./monitoring/grafana/dashboards:/var/lib/grafana/dashboards:ro
""",
    "Dockerfile.detector": """FROM python:3.12-slim

WORKDIR /app

COPY pyproject.toml /app/pyproject.toml
RUN pip install --no-cache-dir uv
RUN uv pip install --system -r pyproject.toml

COPY runtime/detector_service.py /app/detector_service.py

CMD ["python", "detector_service.py"]
""",
    "Dockerfile.app": """FROM python:3.12-slim

WORKDIR /app

COPY pyproject.toml /app/pyproject.toml
RUN pip install --no-cache-dir uv
RUN uv pip install --system -r pyproject.toml

COPY runtime/app_service.py /app/app_service.py

CMD ["python", "app_service.py"]
""",
    "runtime/detector_service.py": """import json
import os
import random
import time

import zenoh
from prometheus_client import Counter, Gauge, Histogram, start_http_server

frames_processed_total = Counter("frames_processed_total", "Total processed frames")
detections_total = Counter("detections_total", "Total detections by class", ["klass"])
corrupt_frames_total = Counter("corrupt_frames_total", "Frames that failed during processing")
inference_duration_seconds = Histogram(
    "inference_duration_seconds",
    "Inference latency in seconds",
    buckets=(0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0),
)
pipeline_duration_seconds = Histogram(
    "pipeline_duration_seconds",
    "Total pipeline latency in seconds",
    buckets=(0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0),
)
last_detection_confidence = Gauge("last_detection_confidence", "Confidence of the last detection")

LITTER_CLASSES = ["bottle", "can", "paper", "plastic_bag", "cigarette", "cup"]


def run_synthetic_inference():
    t0 = time.perf_counter()
    time.sleep(max(0.005, random.gauss(0.04, 0.01)))
    n = random.choices([0, 1, 2, 3], weights=[0.35, 0.4, 0.2, 0.05])[0]

    detections = []
    for _ in range(n):
        klass = random.choice(LITTER_CLASSES)
        conf = max(0.25, min(0.99, random.gauss(0.72, 0.12)))
        detections_total.labels(klass=klass).inc()
        last_detection_confidence.set(conf)
        detections.append({"class": klass, "confidence": round(conf, 3)})

    duration = time.perf_counter() - t0
    inference_duration_seconds.observe(duration)
    return detections, duration


def main():
    prom_port = int(os.getenv("PROMETHEUS_PORT", "9100"))
    start_http_server(prom_port)

    loop_hz = float(os.getenv("DETECTOR_LOOP_HZ", "8"))
    period = 1.0 / max(loop_hz, 0.1)

    model_name = os.getenv("MODEL_NAME", "synthetic-unet")
    router = os.getenv("ZENOH_ROUTER", "localhost:7447")

    conf = zenoh.Config()
    conf.insert_json5("connect/endpoints", json.dumps([f"tcp/{router}"]))
    session = zenoh.open(conf)

    try:
        while True:
            frame_t0 = time.perf_counter()
            try:
                detections, inf_s = run_synthetic_inference()
                payload = {
                    "ts": time.time(),
                    "model": model_name,
                    "latency_ms": round(inf_s * 1000, 2),
                    "detections": detections,
                }
                session.put("litter/detections", json.dumps(payload).encode("utf-8"))
                frames_processed_total.inc()
                pipeline_duration_seconds.observe(time.perf_counter() - frame_t0)
            except Exception:
                corrupt_frames_total.inc()

            elapsed = time.perf_counter() - frame_t0
            if elapsed < period:
                time.sleep(period - elapsed)
    finally:
        session.close()


if __name__ == "__main__":
    main()
""",
    "runtime/app_service.py": """import json
import os
import threading
import time
from collections import deque
from typing import Any

import uvicorn
import zenoh
from fastapi import FastAPI, HTTPException
from prometheus_client import Counter, Gauge, Histogram, start_http_server

api_requests_total = Counter("api_requests_total", "API requests by route", ["route"])
events_received_total = Counter("events_received_total", "Total detection events received")
app_errors_total = Counter("app_errors_total", "Total application errors")
detections_per_event = Histogram(
    "detections_per_event",
    "Number of detections per event",
    buckets=(0, 1, 2, 3, 5, 10),
)
last_event_age_seconds = Gauge("last_event_age_seconds", "Age of last received event in seconds")
last_inference_latency_ms = Gauge("last_inference_latency_ms", "Last seen inference latency in milliseconds")

app = FastAPI(title="Litter Monitoring API")

_latest: dict[str, Any] | None = None
_history = deque(maxlen=500)
_lock = threading.Lock()


def zenoh_consumer():
    global _latest

    router = os.getenv("ZENOH_ROUTER", "localhost:7447")
    conf = zenoh.Config()
    conf.insert_json5("connect/endpoints", json.dumps([f"tcp/{router}"]))
    session = zenoh.open(conf)

    def on_detection(sample):
        global _latest
        try:
            payload = json.loads(bytes(sample.payload))
            n = len(payload.get("detections", []))
            events_received_total.inc()
            detections_per_event.observe(n)
            last_inference_latency_ms.set(float(payload.get("latency_ms", 0.0)))

            with _lock:
                _latest = payload
                _history.append(payload)
        except Exception:
            app_errors_total.inc()

    session.declare_subscriber("litter/detections", on_detection)

    while True:
        with _lock:
            if _latest is None:
                last_event_age_seconds.set(-1)
            else:
                last_event_age_seconds.set(
                    max(0.0, time.time() - float(_latest.get("ts", time.time())))
                )
        time.sleep(1)


@app.get("/health")
def health():
    api_requests_total.labels(route="/health").inc()
    return {"status": "ok"}


@app.get("/api/latest")
def latest():
    api_requests_total.labels(route="/api/latest").inc()
    with _lock:
        if _latest is None:
            raise HTTPException(status_code=404, detail="No events received yet")
        return _latest


@app.get("/api/stats")
def stats():
    api_requests_total.labels(route="/api/stats").inc()

    with _lock:
        items = list(_history)

    total_events = len(items)
    total_detections = sum(len(x.get("detections", [])) for x in items)

    avg_latency = 0.0
    if total_events > 0:
        avg_latency = sum(float(x.get("latency_ms", 0.0)) for x in items) / total_events

    by_class: dict[str, int] = {}
    for ev in items:
        for det in ev.get("detections", []):
            c = det.get("class", "unknown")
            by_class[c] = by_class.get(c, 0) + 1

    return {
        "events": total_events,
        "detections_total": total_detections,
        "avg_latency_ms": round(avg_latency, 2),
        "by_class": by_class,
    }


def main():
    prom_port = int(os.getenv("PROMETHEUS_PORT", "9101"))
    start_http_server(prom_port)

    t = threading.Thread(target=zenoh_consumer, daemon=True)
    t.start()

    app_port = int(os.getenv("APP_PORT", "8000"))
    uvicorn.run(app, host="0.0.0.0", port=app_port)


if __name__ == "__main__":
    main()
""",
    "monitoring/prometheus/prometheus.yml": """global:
  scrape_interval: 5s

scrape_configs:
  - job_name: detector
    static_configs:
      - targets: [\"detector:9100\"]

  - job_name: app
    static_configs:
      - targets: [\"app:9101\"]
""",
    "monitoring/grafana/provisioning/datasources/datasources.yml": """apiVersion: 1

datasources:
  - name: Prometheus
    type: prometheus
    access: proxy
    url: http://prometheus:9090
    isDefault: true
    editable: false
""",
    "monitoring/grafana/provisioning/dashboards/dashboards.yml": """apiVersion: 1

providers:
  - name: Litter Dashboards
    orgId: 1
    folder: Litter
    type: file
    disableDeletion: false
    updateIntervalSeconds: 10
    allowUiUpdates: true
    options:
      path: /var/lib/grafana/dashboards
""",
    "monitoring/grafana/dashboards/litter-overview.json": """{
  \"title\": \"Litter Monitoring Overview\",
  \"timezone\": \"browser\",
  \"schemaVersion\": 38,
  \"version\": 1,
  \"refresh\": \"5s\",
  \"panels\": [
    {
      \"type\": \"stat\",
      \"title\": \"Frames Processed\",
      \"gridPos\": { \"x\": 0, \"y\": 0, \"w\": 6, \"h\": 5 },
      \"targets\": [{ \"expr\": \"sum(frames_processed_total)\" }]
    },
    {
      \"type\": \"stat\",
      \"title\": \"Events Received\",
      \"gridPos\": { \"x\": 6, \"y\": 0, \"w\": 6, \"h\": 5 },
      \"targets\": [{ \"expr\": \"sum(events_received_total)\" }]
    },
    {
      \"type\": \"stat\",
      \"title\": \"Corrupt Frames\",
      \"gridPos\": { \"x\": 12, \"y\": 0, \"w\": 6, \"h\": 5 },
      \"targets\": [{ \"expr\": \"sum(corrupt_frames_total)\" }]
    },
    {
      \"type\": \"timeseries\",
      \"title\": \"Inference p95 (s)\",
      \"gridPos\": { \"x\": 0, \"y\": 5, \"w\": 12, \"h\": 8 },
      \"targets\": [
        {
          \"expr\": \"histogram_quantile(0.95, sum(rate(inference_duration_seconds_bucket[5m])) by (le))\"
        }
      ]
    },
    {
      \"type\": \"timeseries\",
      \"title\": \"Pipeline p95 (s)\",
      \"gridPos\": { \"x\": 12, \"y\": 5, \"w\": 12, \"h\": 8 },
      \"targets\": [
        {
          \"expr\": \"histogram_quantile(0.95, sum(rate(pipeline_duration_seconds_bucket[5m])) by (le))\"
        }
      ]
    },
    {
      \"type\": \"timeseries\",
      \"title\": \"Detections by Class (rate)\",
      \"gridPos\": { \"x\": 0, \"y\": 13, \"w\": 24, \"h\": 8 },
      \"targets\": [
        {
          \"expr\": \"sum by (klass) (rate(detections_total[5m]))\"
        }
      ]
    }
  ]
}
""",
    "docs/monitoring.md": """# Monitoring Setup

## Ziel
Runtime-Monitoring fuer die Litter-Detection-Pipeline mit Dashboards,
getrennt vom Training-Tracking in MLflow.

## Start
1. uv sync
2. docker compose -f docker-compose.monitoring.yml up --build

## URLs
- Grafana: http://localhost:3000
- Prometheus: http://localhost:9090
- API health: http://localhost:8000/health
- API latest: http://localhost:8000/api/latest
- API stats: http://localhost:8000/api/stats

## Beobachtete Metriken
- frames_processed_total
- detections_total nach Klasse
- inference_duration_seconds
- pipeline_duration_seconds
- corrupt_frames_total
- events_received_total
- last_event_age_seconds

## Bezug zur Aufgabenstellung
- MLflow: Trainings- und Experimenttracking
- Monitoring-Stack: Laufzeitbeobachtung der Anwendung inkl. Dashboards
""",
}

PYPROJECT_NOTE = """
Manueller Schritt fuer pyproject.toml (dependencies):

- fastapi>=0.115.0
- uvicorn>=0.30.0
- prometheus-client>=0.24.0
- eclipse-zenoh>=1.0.0
"""


def write_files(force: bool = False) -> None:
    for rel_path, content in FILES.items():
        target = PROJECT_ROOT / rel_path
        target.parent.mkdir(parents=True, exist_ok=True)

        if target.exists() and not force:
            print(f"SKIP (exists): {rel_path}")
            continue

        target.write_text(content, encoding="utf-8")
        print(f"WROTE: {rel_path}")


print("=== Monitoring Blueprint ===")
print(PYPROJECT_NOTE.strip())
print()
print("Templates loaded:", len(FILES))
for rel_path in FILES:
    print("-", rel_path)

In [ ]:
# Schreibe alle Dateien (ohne Ueberschreiben vorhandener Dateien)
write_files(force=False)
print('Fertig. Wenn noetig mit force=True erneut ausfuehren.')

In [ ]:
# Optional: Branch nochmal direkt im Notebook pruefen
import subprocess

branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
print('Current branch:', branch)

## Start des Stacks

Nach dem Schreiben der Dateien und `uv sync`:

```bash
docker compose -f docker-compose.monitoring.yml up --build
```

Dann pruefen:
- Grafana: http://localhost:3000
- Prometheus: http://localhost:9090
- API Health: http://localhost:8000/health